# RobustWorld: post-train V-JEPA 2 on Colab

Fine-tunes V-JEPA 2's **predictor** on your clips so its imagined future can learn the blockade,
then scores it against the pretrained model with the same frozen evaluation decoder
(see `CLAUDE.md`). The encoder stays frozen, so its features are computed once and cached.

1. Upload your clips (a zip from `python scripts/pack_clips.py` on your computer).
2. Encode every clip once (frozen encoder).
3. Score the **pretrained** predictor (the "before").
4. Post-train one predictor per cross-validation fold.
5. Score the **post-trained** predictors (the "after") and draw the blocker figures.

**Runtime:** GPU. An **A100** or **L4** is best (bf16). A **T4** works (fp16) but is slower.

Everything stays in this Colab session; the last cell downloads the results as a zip.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/RobustWorld"
BRANCH = "claude/project-thread-qvh5by"   # switch to "main" once this is merged
if not os.path.exists(REPO):
    try:  # only needed if the repo is private again
        url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/FinPThomas/RobustWorld.git"
    except Exception:
        url = "https://github.com/FinPThomas/RobustWorld.git"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, REPO], check=True, capture_output=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    "https://github.com/FinPThomas/RobustWorld.git"], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -e . -r models/vjepa2/requirements.txt

## 1. Upload the clips

On your computer, after `robustworld` has cut the clips, run `python scripts/pack_clips.py` and
upload `outputs/robustworld_clips.zip` here. It holds the clips, manifests and ball tracking.

In [ ]:
import glob, zipfile
from google.colab import files
zips = glob.glob("/content/robustworld_clips*.zip") + glob.glob(f"{REPO}/robustworld_clips*.zip")
if not zips:
    zips = list(files.upload())            # saved in the current folder (the repo)
zipfile.ZipFile(zips[0]).extractall(REPO)
!wc -l data/processed/clips/manifest.jsonl

## 2. Encode every clip once

Runs the frozen encoder over each clip (full clip and context half alone) and caches the
features in `outputs/vjepa2/cache/`. Clips that are already cached are skipped.

In [ ]:
!python models/vjepa2/posttrain.py encode

## 3. Before: the pretrained predictor

Cross-validated ball readout and the blocker figures for the pretrained model.

In [ ]:
!python models/vjepa2/ball_probe_cv.py
!python models/vjepa2/blocker_figs.py --label pretrained --out outputs/vjepa2/blocker/pretrained

## 4. The experiment grid: before and after post-training

Four ways to run V-JEPA's predictor, each scored **before** (the pretrained predictor, 0 epochs)
and **after** post-training, with the same frozen decoder and folds:

| variant | what changes |
|---|---|
| `plain` | V-JEPA's own latent L1, all future steps at once (the baseline) |
| `codes` | predicts one of 256 discrete feature codes per token (cross-entropy), snapped at inference, so it can't output a blur |
| `rollout` | predicts one step at a time and feeds its own prediction back in |
| `codes_rollout` | both |

The codebook is k-means on each fold's own training features, so it learns nothing from outcomes.
Try `--variants plain --extra --fold 0 --epochs 2` first to check timing.

In [ ]:
EPOCHS = 10
VARIANTS = "plain codes rollout codes_rollout"
!python models/vjepa2/experiments.py run --epochs {EPOCHS} --variants {VARIANTS}

## 5. Results

Headline: **P(correct)**, the mean probability given to the true outcome (through, bounce,
hidden) with each outcome weighted equally, and **balanced accuracy**. Plus **ball hit rate**: how
often the imagined ball is within 48 px of the real one. The dashed line is the decoder on the
real future frames, the ceiling.

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("outputs/vjepa2/experiments/summary.md").read()))
display(Image("outputs/vjepa2/experiments/summary.png"))

### Blocker figure for one run

P(blocked) against where the ball meets the plank, after (solid) over before (dashed).

In [ ]:
V = "codes"
for phase in ("before", "after"):
    extra = f"--before outputs/vjepa2/blocker/{V}-before/crossing.json" if phase == "after" else ""
    !python models/vjepa2/blocker_figs.py --per-clip outputs/vjepa2/ball_probe_cv/{V}-{phase}/per_clip.json \
        --label {V}-{phase} --out outputs/vjepa2/blocker/{V}-{phase} {extra}
display(Image(f"outputs/vjepa2/blocker/{V}-after/blocked_vs_crossing.png"))

## Download results

Zips the scores, figures and training logs (not the feature cache). Set `WITH_CHECKPOINTS = True`
to include the post-trained predictors too (about 90 MB per fold).

In [ ]:
import shutil, pathlib
WITH_CHECKPOINTS = False
out = pathlib.Path("/content/robustworld_posttrain")
shutil.rmtree(out, ignore_errors=True)
shutil.copytree("outputs/vjepa2", out / "outputs/vjepa2", ignore=shutil.ignore_patterns("cache"))
ck = pathlib.Path("checkpoints/vjepa2")
shutil.copytree(ck, out / "checkpoints/vjepa2", ignore=None if WITH_CHECKPOINTS else shutil.ignore_patterns("*.pt"))
files.download(shutil.make_archive(str(out), "zip", out))